# RAG (Retrieval-Augmented Generation) — Step-by-Step Demo

**What is RAG?**  
Large Language Models (LLMs) are trained on general internet data and have a knowledge cutoff date.  
They cannot answer questions about *your* documents — your company policy, your product manual, your internal KB.

**RAG solves this** by combining two steps:
1. **Retrieve** — Find the most relevant passages from your documents for the user's question
2. **Generate** — Feed those passages as context to the LLM so it can answer accurately

```
┌─────────────┐     embed      ┌──────────────────┐
│  Your PDF   │ ─────────────► │  Vector Store    │
│  (chunks)   │                │  (Chroma DB)     │
└─────────────┘                └────────┬─────────┘
                                        │  similarity search
┌─────────────┐     embed               ▼
│  Question   │ ─────────────► top-k relevant chunks
└─────────────┘                         │
                                        ▼
                             ┌──────────────────────┐
                             │  LLM (Qwen3 / Groq)  │
                             │  context + question  │
                             └──────────┬───────────┘
                                        │
                                        ▼
                                     Answer
```

**Source used:** A single telecom reference guide PDF (8 sections covering mobile networks, SIM, billing, roaming, security, etc.)

## Setup

Install dependencies (run once, then restart the kernel):

In [6]:
# Uncomment and run once to install dependencies
!pip install langchain langchain-community langchain-chroma langchain-groq langchain-huggingface pypdf sentence-transformers python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: C:\Users\narah\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [7]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env file in this folder

# Or set it directly here for quick demos:
# os.environ["GROQ_API_KEY"] = "your-key-here"

assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in a .env file or above"
print("API key loaded.")

API key loaded.


---
## Step 1 — Load the PDF

`PyPDFLoader` reads every page of the PDF and returns a list of `Document` objects.  
Each `Document` has a `.page_content` string (the raw text of that page).

At this stage, pages are often too long to embed efficiently — that's what Step 2 fixes.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"  # adjust if needed

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

C:\Users\narah\AppData\Local\Temp\ipykernel_29356\1501032524.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\work\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


---
## Step 2 — Split into Chunks

**Why chunk?**  
- Embedding models have a token limit (e.g. 512 tokens for all-MiniLM)
- Smaller, focused chunks → better semantic similarity scores
- We want to retrieve the *relevant paragraph*, not a whole page

**Key parameters:**
- `chunk_size` — max characters per chunk
- `chunk_overlap` — characters shared between neighbouring chunks (prevents losing context at boundaries)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # ~150 words per chunk
    chunk_overlap=100,    # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "],  # tries paragraph → line → sentence → word
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}  (from {len(pages)} pages)")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n--- Example chunk ---")
print(chunks[5].page_content)

Total chunks: 37  (from 9 pages)
Avg chunk length: 504 chars

--- Example chunk ---
Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.


---
## Step 3 — Embed Chunks and Store in a Vector Database

**What is an embedding?**  
A numerical vector (list of floats) that represents the *meaning* of a piece of text.  
Texts with similar meaning have vectors that are close together in high-dimensional space.

**What is a vector store?**  
A database optimised for *similarity search* — given a query vector, find the nearest stored vectors.

Here we use:
- **Embedding model:** `all-MiniLM-L6-v2` — a small, fast, open-source sentence transformer (384 dimensions)
- **Vector store:** Chroma — an in-memory (or persisted) vector DB

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma (in memory)...")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading embedding model (downloads ~90 MB on first run)...


C:\work\tutorial-agentic-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\narah\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7913.93it/s]


Embedding all chunks and storing in Chroma (in memory)...
Vector store ready. 37 vectors stored.


---
## Step 4 — Test the Retriever

Before hooking up the LLM, let's verify the retriever works.  
It embeds the query with the same model, then finds the top-k nearest chunk vectors.

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call c

---
## Step 5 — Build the RAG Chain

Now we wire everything together using LangChain's **LCEL (LangChain Expression Language)**:

```
question
   │
   ├──► retriever ──► join chunks into one string ──►  context
   │                                                        │
   └────────────────────────────────────────────►  prompt template
                                                            │
                                                           LLM
                                                            │
                                                       Answer (string)
```

The system prompt tells the LLM to *only* use the retrieved context — this prevents hallucination.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

---
## Step 6 — Ask a Single Question

Let's test the full pipeline end-to-end with one hard-coded question:

In [ ]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

---
## Step 7 — Interactive Q&A Loop

Run this cell and keep asking questions.  
Type `quit` to exit.

> **Try these questions:**
> - "What is the difference between 4G and 5G?"
> - "How do I fix a SIM card that is not being detected?"
> - "What is SIM swap fraud and how can I protect myself?"
> - "Explain VoLTE in simple terms."
> - "What happens during a billing dispute?"

In [ ]:
print("Telecom RAG Assistant — type 'quit' to exit\n")

while True:
    question = input("Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    print("\nAnswer:")
    for chunk in chain.stream(question):
        print(chunk, end="", flush=True)
    print("\n")

---
## Bonus — Inspect What Was Retrieved

This cell shows you exactly which chunks were retrieved for a given question, before the LLM sees them.  
Great for debugging retrieval quality.

In [ ]:
debug_question = "What security measures protect against SIM swap fraud?"

docs = retriever.invoke(debug_question)
print(f"Question: {debug_question}")
print(f"Retrieved {len(docs)} chunks:\n")
for i, doc in enumerate(docs, 1):
    print(f"{'='*60}")
    print(f"Chunk {i} (page {doc.metadata.get('page', '?')})")
    print(f"{'='*60}")
    print(doc.page_content)
    print()

print("\nFinal Answer:")
print(chain.invoke(debug_question))